In [ ]:
# 04_evaluation.ipynb
# Goal of this notebook:
# - Load the trained models from notebook 03
# - Load the held-out test set (same for all models)
# - Compute test metrics: AUC (+ CI), accuracy, sensitivity, specificity
# - Save metrics into a table for the report
# - Optionally:
#    - Plot ROC curves
#    - Statistically compare models (paired tests on probabilities)
#
# You should:
# - Understand what each metric tells you in a medical context
# - Keep the test set "holy" (only used here, after training is done)

***

## 0. Setup and imports

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    auc,
    accuracy_score,
    confusion_matrix,
    recall_score,
    f1_score,
    precision_score,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_rel, norm
import pickle

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)

# 0.1 Set project root to the repository root (where the .pkl and .csv files live)
REPOROOT = Path(os.path.abspath(''))
# If running from a subdirectory, walk up to find the data files
for _candidate in [REPOROOT, REPOROOT.parent, Path('/home/runner/work/Ineke/Ineke')]:
    if (_candidate / 'lr_raw_binary_centers.pkl').exists():
        REPOROOT = _candidate
        break

MODELDIR = REPOROOT  # models (.pkl) live in the repo root
DATADIR  = REPOROOT  # data files (.csv, .xlsx) live in the repo root
RESULTS  = REPOROOT  # output files also go to repo root

print('Repository root:', REPOROOT)
print('PKL files found:', [p.name for p in sorted(REPOROOT.glob('*.pkl'))])


def sensitivity_score(y_true, y_pred, pos_label=1):
    """Convenience wrapper for sensitivity (recall of the positive class)."""
    return recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0)


def specificity_score(y_true, y_pred, neg_label=0):
    """Convenience wrapper for specificity (recall of the negative class)."""
    return recall_score(y_true, y_pred, pos_label=neg_label, zero_division=0)


print('\u2713 Setup complete')


**Student direction:**

- Ensure the same environment/kernel is used as for training, otherwise pickled models may fail to load.

***

## 1. Load trained models and test set

We now load:

- All `*.pkl` models in `results/models/` (from notebook 03).
- Test features and labels saved there as `.npy`.

In [ ]:
# 1.1 Load models from repository root
models = {}

for model_file in sorted(MODELDIR.glob('*.pkl')):
    model_name = model_file.stem
    with open(model_file, 'rb') as f:
        models[model_name] = pickle.load(f)

print(f'\u2713 Loaded {len(models)} models')
print(' Models:', ', '.join(sorted(models.keys())))

# 1.2 Load radiomics features, center labels, and top-20 feature list
features_df = pd.read_csv(DATADIR / 'features_raw (1).csv')
center_df   = pd.read_excel(DATADIR / 'center_ids.xlsx')     # columns: case, center
top20_df    = pd.read_csv(DATADIR / 'top20_features (1).csv') # columns: feature, stability_score
feature_names = top20_df['feature'].tolist()

print(f'Features table: {features_df.shape}')
print(f'Center labels:  {center_df.shape} | unique centers: {sorted(center_df["center"].unique())}')
print(f'Top-20 feature names loaded: {len(feature_names)}')

# 1.3 Merge features with center labels
merged = features_df[['caseid'] + feature_names].merge(
    center_df, left_on='caseid', right_on='case', how='inner'
)
merged = merged.dropna(subset=['center'])

X_raw    = merged[feature_names].to_numpy()
case_ids = merged['caseid'].to_numpy()
y_raw    = merged['center'].to_numpy()

# 1.4 Map centers to binary labels: center 1 -> 0, center 2 -> 1
y_aligned = np.where(y_raw == 2, 1, 0)

print(f'\nTotal samples: {len(y_aligned)}')
print(f'Class distribution: center-1 (label 0)={np.sum(y_aligned==0)}, '
      f'center-2 (label 1)={np.sum(y_aligned==1)}')

# 1.5 Reproduce the center-stratified 70 / 15 / 15 split (same random_state=42)
idx = np.arange(len(y_aligned))
train_idx, temp_idx, y_train, y_temp = train_test_split(
    idx, y_aligned, test_size=0.30, random_state=42, stratify=y_aligned
)
val_idx, test_idx, y_val, y_test = train_test_split(
    temp_idx, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'\nTrain: {len(train_idx)} samples ({len(train_idx)/len(y_aligned)*100:.1f}%)')
print(f'Val:   {len(val_idx)} samples ({len(val_idx)/len(y_aligned)*100:.1f}%)')
print(f'Test:  {len(test_idx)} samples ({len(test_idx)/len(y_aligned)*100:.1f}%)')

# 1.6 Standardise features (fit on training set only, apply to all sets)
scaler = StandardScaler()
scaler.fit(X_raw[train_idx])

X_raw_train = scaler.transform(X_raw[train_idx])
X_raw_val   = scaler.transform(X_raw[val_idx])
X_raw_test  = scaler.transform(X_raw[test_idx])

print(f'\n\u2713 Test data ready:  X_raw_test.shape = {X_raw_test.shape}, '
      f'y_test.shape = {y_test.shape}')
print(f'  Positive class in test: {y_test.sum()} ({np.mean(y_test):.1%})')


**Student direction:**

- Confirm that the number of models matches what you trained (e.g. `lr_raw_binary_centers`, `rf_raw_binary_centers`, `gb_raw_binary_centers`).
- Confirm that `X_raw_test.shape[^9_0] == len(y_test)`.

***

## 2. AUC with confidence intervals (Hanley–McNeil approximation)

This helper computes AUC and an approximate confidence interval.

In [ ]:
def compute_auc_ci(y_true, y_pred_proba, alpha=0.05):
    """
    Compute AUC with a (1-alpha) confidence interval using the Hanley–McNeil approximation.

    Parameters:
        y_true: array of true binary labels (0/1)
        y_pred_proba: array of predicted probabilities for the positive class
        alpha: significance level (0.05 -> 95% CI)

    Returns:
        auc_score, ci_low, ci_high
    """
    auc_score = roc_auc_score(y_true, y_pred_proba)

    n_pos = np.sum(y_true)
    n_neg = len(y_true) - n_pos

    if n_pos == 0 or n_neg == 0:
        return auc_score, np.nan, np.nan

    q1 = auc_score / (2 - auc_score)
    q2 = 2 * auc_score**2 / (1 + auc_score)

    var = (
        auc_score * (1 - auc_score)
        + (n_pos - 1) * (q1 - auc_score**2)
        + (n_neg - 1) * (q2 - auc_score**2)
    ) / (n_pos * n_neg)

    var = max(var, 0.0)
    se = np.sqrt(var)

    z = norm.ppf(1 - alpha / 2)
    ci_low = max(0.0, auc_score - z * se)
    ci_high = min(1.0, auc_score + z * se)

    return auc_score, ci_low, ci_high

**Student direction:**

- You don’t need to edit this function; you will call it in the main evaluation loop.

***

## 3. Evaluate all models on the test set

For each model, we compute:

- AUC + CI
- Accuracy
- Sensitivity, specificity
- Store per-case probabilities for later ROC and statistical tests.

In [ ]:
results = []

for model_name, model in sorted(models.items()):
    X_test = X_raw_test

    # 3.1 Predicted probabilities and hard labels
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred       = model.predict(X_test)

    # Ensure labels are in {0, 1}
    y_pred_bin = np.where(y_pred == 2.0, 1, y_pred).astype(int)

    # 3.2 Core metrics
    auc_score, ci_low, ci_high = compute_auc_ci(y_test, y_pred_proba)
    accuracy  = accuracy_score(y_test, y_pred_bin)

    cm = confusion_matrix(y_test, y_pred_bin)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # 3.3 Extended metrics (precision, recall, F1, average-precision / PR-AUC)
    precision = precision_score(y_test, y_pred_bin, zero_division=0)
    recall    = recall_score(y_test, y_pred_bin, zero_division=0)
    f1        = f1_score(y_test, y_pred_bin, average='macro')
    avg_prec  = average_precision_score(y_test, y_pred_proba)  # area under PR curve

    results.append(
        {
            'model':               model_name,
            'auc':                 auc_score,
            'auc_ci_low':          ci_low,
            'auc_ci_high':         ci_high,
            'accuracy':            accuracy,
            'sensitivity':         sensitivity,
            'specificity':         specificity,
            'precision':           precision,
            'recall':              recall,
            'f1_macro':            f1,
            'avg_precision_score': avg_prec,
            'y_pred_proba':        y_pred_proba,  # kept for ROC / t-test
        }
    )

# 3.4 Display clean table (without probability arrays)
display_cols = ['model', 'auc', 'auc_ci_low', 'auc_ci_high', 'accuracy',
                'sensitivity', 'specificity', 'precision', 'recall', 'f1_macro',
                'avg_precision_score']
results_display = pd.DataFrame(
    [{k: v for k, v in r.items() if k in display_cols} for r in results]
)[display_cols]

print('\n' + '=' * 90)
print('TEST SET PERFORMANCE METRICS')
print('=' * 90)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
print(results_display.to_string(index=False))

# 3.5 Save metrics table
results_display.to_csv(RESULTS / 'evaluation_metrics.csv', index=False)
print('\n\u2713 Saved evaluation_metrics.csv to', RESULTS)

# 3.6 Precision-Recall curves
fig, ax = plt.subplots(figsize=(7, 6))
for r in results:
    prec_arr, rec_arr, _ = precision_recall_curve(y_test, r['y_pred_proba'])
    ax.plot(rec_arr, prec_arr, label=f"{r['model']} (AP={r['avg_precision_score']:.3f})")
# Baseline: fraction of positives
ax.axhline(y=y_test.mean(), color='k', linestyle='--', label=f'No-skill (AP={y_test.mean():.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall curves on test set')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print('\u2713 Evaluation complete')


**Student direction:**

- Inspect the printed table:
    - Which model has the highest AUC?
    - How do sensitivity and specificity compare?
- Remember that in medicine, sensitivity vs specificity trade‑offs matter, not just accuracy.

***

## 4. Confusion matrix and probability inspection for a reference model

It can be helpful to look closely at one model (e.g. logistic regression) to see how it behaves.

In [ ]:
ref_model_name = 'lr_raw_binary_centers'  # reference model for close inspection

cm = confusion_matrix(y_test, models[ref_model_name].predict(X_raw_test))
print(f'Confusion matrix for {ref_model_name}:')
print(cm)
tn_r, fp_r, fn_r, tp_r = cm.ravel()
print(f'  TP={tp_r}  FP={fp_r}  TN={tn_r}  FN={fn_r}')

probs_pos = models[ref_model_name].predict_proba(X_raw_test)[:, 1]
print(f'\nPredicted probability stats:')
print(f'  Min:    {probs_pos.min():.4f}')
print(f'  Median: {np.median(probs_pos):.4f}')
print(f'  Max:    {probs_pos.max():.4f}')
if y_test.sum() > 0:
    print(f'  Max prob among true positives: {probs_pos[y_test == 1].max():.4f}')

# Visualise confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred Center1', 'Pred Center2'],
            yticklabels=['True Center1', 'True Center2'])
ax.set_title(f'Confusion matrix: {ref_model_name}')
plt.tight_layout()
plt.show()


**Student direction:**

- Use the confusion matrix to compute by hand: TP, FP, TN, FN.
- Reflect: are there many false negatives (missed center 2) or false positives?

***

## 5. ROC curves for all models on the test set

We now visualize ROC curves for each model to compare their discrimination.

In [ ]:
def plot_roc_curves(results, y_test):
    """
    Plot ROC curves for each model stored in `results`.
    """
    plt.figure(figsize=(7, 6))

    for r in results:
        model_name = r["model"]
        y_pred_proba = r["y_pred_proba"]
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{model_name} (AUC = {roc_auc:.3f})")

    # Chance line
    plt.plot([0, 1], [0, 1], "k--", label="Chance")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC curves on test set")
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()
# Call the ROC-curve plot
plot_roc_curves(results, y_test)


**Student direction:**

- Call `plot_roc_curves(results, y_test)` once you have the `results` list.
- Notice which curve is furthest towards the top‑left corner (better).

***

## 6. Optional: Paired statistical comparison of models

We can use paired tests on predicted probabilities to ask whether AUC differences might be due to chance. A more rigorous approach uses bootstrapping, but here’s a simple paired t‑test scaffold.

In [ ]:
def compare_models_paired_ttest(results, model_ids=None, alpha=0.05):
    """
    Paired t-tests on per-case predicted probabilities between models.

    Parameters:
        results: list of dicts (as above), each with keys 'model' and 'y_pred_proba'
        model_ids: list of model names to compare; if None, pick three standard ones
        alpha: global significance level (Bonferroni correction applied)
    """
    # Build a dict for easy lookup
    model_info = {r["model"]: r for r in results}

    if model_ids is None:
        # Example default: three models used in notebook 03
        model_ids = ["lr_raw_binary_centers", "rf_raw_binary_centers", "gb_raw_binary_centers"]

    # Sanity check: all present
    missing = [m for m in model_ids if m not in model_info]
    if missing:
        print("Missing models in results:", missing)
        return

    pairs = [
        (model_ids[0], model_ids[1]),
        (model_ids[0], model_ids[2]),
        (model_ids[1], model_ids[2]),
    ]

    alpha_pairwise = alpha / len(pairs)
    print("=" * 80)
    print("STATISTICAL COMPARISON (paired t-tests on predicted probabilities)")
    print(f" Global alpha = {alpha}, pairwise alpha (Bonferroni) = {alpha_pairwise:.4f}")
    print("=" * 80)

    rows = []

    for m1, m2 in pairs:
        r1 = model_info[m1]
        r2 = model_info[m2]

        auc1 = r1["auc"]
        auc2 = r2["auc"]

        ypred1 = r1["y_pred_proba"]
        ypred2 = r2["y_pred_proba"]

        tstat, pvalue = ttest_rel(ypred1, ypred2)

        print(f"{m1} vs {m2}")
        print(f" {m1} AUC: {auc1:.4f}")
        print(f" {m2} AUC: {auc2:.4f}")
        print(f" AUC difference (m2 - m1): {auc2 - auc1:.4f}")
        print(f" t-statistic: {tstat:.4f}")
        print(f" p-value: {pvalue:.4f} {'(significant)' if pvalue < alpha_pairwise else '(n.s.)'}")
        print()

        rows.append(
            {
                "model1": m1,
                "model2": m2,
                "auc1": auc1,
                "auc2": auc2,
                "auc_diff_m2_minus_m1": auc2 - auc1,
                "tstat": tstat,
                "pvalue": pvalue,
                "significant_bonferroni": "Yes" if pvalue < alpha_pairwise else "No",
            }
        )

    stats_df = pd.DataFrame(rows)
    stats_df.to_csv(RESULTS / "statistical_tests_models.csv", index=False)
    print("Saved statistical comparison table to:", RESULTS / "statistical_tests_models.csv")
# Call the statistical comparison
compare_models_paired_ttest(results)


**Student direction:**

- Use this only if you want to comment on whether AUC differences are likely to be due to chance.
- Remember that statistical significance is not the same as clinical importance.

***

## 7. Optional: Calibration plot for one model

Calibration shows whether predicted probabilities match observed frequencies.

In [ ]:
def plot_calibration(model_name, model, X_test, y_test, n_bins=10):
    """
    Reliability plot (calibration curve) for one model.
    """
    prob_pos = model.predict_proba(X_test)[:, 1]
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_test, prob_pos, n_bins=n_bins, strategy="uniform"
    )

    plt.figure(figsize=(6, 6))
    plt.plot(mean_predicted_value, fraction_of_positives, "s-", label=model_name)
    plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")

    plt.xlabel("Mean predicted probability")
    plt.ylabel("Fraction of positives")
    plt.title(f"Calibration curve: {model_name}")
    plt.legend(loc="upper left")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()
# Call calibration plot for the logistic regression model
plot_calibration('lr_raw_binary_centers', models['lr_raw_binary_centers'], X_raw_test, y_test)


**Student direction:**

- Choose one model (e.g. best AUC) and call `plot_calibration(model_name, model, X_raw_test, y_test)`.
- Interpret whether the model tends to over‑ or under‑estimate risk.

***

## 8. Checklist for this evaluation notebook

By the end of `04_evaluation.ipynb`, you should have:

- A table `results/evaluation_metrics.csv` with AUC (+ CI), accuracy, sensitivity, and specificity for all models.
- ROC curves showing which model discriminates best.
- Optionally, a `results/statistical_tests_models.csv` file with paired tests between models.
- One or two figures (e.g. ROC and calibration) and a short interpretation you can reuse in your final report.

[^9_1]: 04_evaluation.ipynb

## 8. Checklist for this evaluation notebook

By the end of `04_evaluation.ipynb`, you should have:

- [x] Loaded **three trained models** (`lr`, `rf`, `gb`) from `.pkl` files in the repository root.
- [x] Reconstructed the **held-out test set** from the raw feature CSV + center labels XLSX using a reproducible center-stratified 70/15/15 split (`random_state=42`).
- [x] Computed test metrics: **AUC (+ 95 % CI)**, accuracy, sensitivity, specificity, precision, recall, macro-F1, and average precision score.
- [x] Plotted **ROC curves** showing which model discriminates best.
- [x] Ran **paired t-tests** (Bonferroni-corrected) to check whether AUC differences are statistically significant.
- [x] Plotted a **calibration curve** for the logistic regression model.

> **Note on class imbalance:** The dataset has ~79 % center-1 vs ~21 % center-2 samples.
> The models were trained with `class_weight='balanced'` / `balanced_subsample` to mitigate this.
> An alternative is SMOTE (synthetic oversampling) applied **inside** the training pipeline only.
